# 07 动量情绪联合策略

## 本课学习目标

- A. 量化金融主线：投资组合（Portfolio）、交易费用（Transaction Cost）和滑点（Slippage）
零基础解释：组合是一篮子资产，费用和滑点会降低回测收益。
- B. 大语言模型主线：大语言模型提供商（LLM Provider）
零基础解释：本课让 Mock Provider 只提供情绪因子，不让模型直接交易。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：生成动量情绪联合权重，运行独立教学回测并输出指标。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 基准（Benchmark）

零基础解释：基准是用来判断策略表现是否真正有价值的参照对象。策略赚钱不代表策略优秀——如果市场整体上涨了 30%，而你的策略只赚了 10%，那这个策略实际上跑输了市场。

### 两个教学基准

本课实现两个最简单的基准：

**1. 等权买入并持有（Equal-Weight Buy and Hold）**
- 初始时对 5 只虚拟股票各分配 20% 资金（等权）；
- 之后不主动调仓，任由持仓随价格波动；
- 使用与策略相同的初始资金和价格区间；
- 初次建仓时计算交易费用（与策略一致）；
- 这个基准回答："什么都不做"和"主动调仓"哪个更好？

**2. 现金基准（Cash Benchmark）**
- 初始资金始终保持为现金，不买入任何资产；
- 净值保持不变（忽略利息）；
- 这个基准回答：至少你的策略有没有"赚钱"（净值 > 1）？

### 如何解读基准对比

| 情况 | 含义 |
|------|------|
| 策略赚钱且跑赢基准 | 策略在绝对和相对意义上都有正收益 |
| 策略赚钱但跑输基准 | 策略有正收益，但不如简单买入持有，可能不值得承担调仓成本 |
| 策略亏损但比基准少亏 | 策略有防御价值，在市场下跌时保护了资金 |
| 策略亏损且比基准多亏 | 策略在绝对和相对意义上都表现不佳 |

**重要：收益为正 ≠ 策略有效。** 必须与基准比较才能判断策略是否真正创造了价值。

下面代码同时运行策略和两个基准，并在一张图上对比。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from learning.src.plotting import configure_chinese_plotting, safe_title
from learning.src.market_data import load_price_data
from learning.src.momentum_factor import compute_momentum, rank_momentum
from learning.src.sentiment_factor import classify_news, daily_sentiment_factor
from learning.src.time_alignment import monthly_signal_schedule
from learning.src.mini_backtest import run_backtest
from learning.src.financial_metrics import (
    simple_returns, annualized_return, annualized_volatility,
    maximum_drawdown, sharpe_ratio,
)

# 配置中文字体（不可用时回退英文）
_font = configure_chinese_plotting()

prices = load_price_data(DATA / "sample_prices.csv")
news = classify_news(pd.read_csv(DATA / "sample_news.csv"))
schedule = monthly_signal_schedule(prices).head(8)
mom = compute_momentum(prices, 20)
tickers_all = sorted(prices["ticker"].unique())

# ---- 动量情绪联合策略 ----
targets = []
for _, s in schedule.iterrows():
    m = rank_momentum(mom, s["signal_date"], 20)
    sent = daily_sentiment_factor(news, s["signal_timestamp"]).groupby("ticker")["sentiment_score"].mean()
    m["sentiment_score"] = m["ticker"].map(sent).fillna(0.0)
    m["combined_score"] = 0.7 * m["momentum_rank_score"] + 0.3 * ((m["sentiment_score"] + 1) / 2)
    for ticker in m.nlargest(2, "combined_score")["ticker"]:
        targets.append({"execution_date": s["execution_date"], "ticker": ticker, "weight": 0.5})
targets = pd.DataFrame(targets)
result_strategy = run_backtest(prices, targets, transaction_cost=0.001, slippage=0.0005)
eq_s = result_strategy.equity_curve.set_index("date")["equity"]

# ---- 等权买入并持有基准 ----
initial_date = schedule["execution_date"].iloc[0]
bh_targets = pd.DataFrame([
    {"execution_date": initial_date, "ticker": t, "weight": 1.0 / len(tickers_all)}
    for t in tickers_all
])
result_bh = run_backtest(prices, bh_targets, transaction_cost=0.001, slippage=0.0005)
eq_bh = result_bh.equity_curve.set_index("date")["equity"]

# ---- 现金基准 ----
eq_cash = pd.Series(1.0, index=eq_s.index, name="cash")

# ---- 计算指标 ----
def compute_metrics(eq_series, label):
    rets = simple_returns(eq_series)
    return {
        "策略": label,
        "累计收益率": eq_series.iloc[-1] / eq_series.iloc[0] - 1,
        "年化收益率": annualized_return(rets),
        "年化波动率": annualized_volatility(rets),
        "最大回撤": maximum_drawdown(eq_series),
        "夏普比率": sharpe_ratio(rets),
    }

all_metrics = pd.DataFrame([
    compute_metrics(eq_s, "动量情绪联合"),
    compute_metrics(eq_bh, "等权买入持有"),
    compute_metrics(eq_cash, "现金基准"),
])
display(all_metrics)

# ---- 净值曲线对比 ----
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eq_s.index, eq_s / eq_s.iloc[0], label="动量情绪联合策略", linewidth=1.5)
ax.plot(eq_bh.index, eq_bh / eq_bh.iloc[0], label="等权买入持有基准", linewidth=1.5, linestyle="--")
ax.axhline(y=1, color="gray", linewidth=0.8, linestyle=":", label="现金基准")
safe_title(ax, "策略净值 vs 基准（合成教学数据）", "Strategy NAV vs Benchmarks (synthetic data)")
ax.set_xlabel("日期")
ax.set_ylabel("净值（初始=1）")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ---- 解释 ----
print("=" * 50)
print("如何解读：")
strat_ret = eq_s.iloc[-1] / eq_s.iloc[0] - 1
bh_ret = eq_bh.iloc[-1] / eq_bh.iloc[0] - 1
if strat_ret > max(bh_ret, 0):
    print("策略赚钱且跑赢基准——策略在绝对和相对意义上都有正收益。")
elif strat_ret > 0 >= bh_ret:
    print("策略赚钱且跑赢基准——策略有正收益而基准亏损。")
elif strat_ret > 0:
    print("策略赚钱但跑输基准——正收益但不如简单买入持有，需考虑调仓成本是否值得。")
elif strat_ret > bh_ret:
    print("策略亏损但比基准少亏——策略有防御价值。")
else:
    print("策略亏损且比基准多亏——在绝对和相对意义上都表现不佳。")
print("注意：以上结论仅针对这段合成教学数据，不能推广到真实市场。")
print("=" * 50)

## 结尾总结

你现在应该理解：策略必须与基准对比才能判断是否有效。

本课核心收获：
- 投资组合由多只资产构成，交易费用和滑点会降低收益；
- Mock Provider 只提供情绪因子，模型不直接参与交易决策；
- 等权买入并持有是"什么都不做"的对照；
- 现金基准是"不参与市场"的对照；
- 策略赚钱 ≠ 策略有效，必须与基准对比；
- 策略亏损但少亏于基准，也有防御价值。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：
- Portfolio（投资组合）
- Transaction Cost（交易费用）
- Slippage（滑点）
- Benchmark（基准）
- Equal-Weight Buy and Hold（等权买入并持有）
- Cash Benchmark（现金基准）
- LLM Provider（大语言模型提供商）

### 常见错误

1. **只看策略收益，不与基准比较**：策略赚了 10% 但市场涨了 30%，策略其实是跑输的。
2. **忽略交易费用**：频繁调仓的费用会显著侵蚀收益，回测必须计入。
3. **权重之和超过 100%**：如果各持仓权重之和 > 1，意味着使用了杠杆，风险计算会失真。
4. **用不同时间段对比策略和基准**：必须使用完全相同的日期范围才有可比性。

### 课后练习

1. **修改动量权重**：将动量权重从 0.7 改为 0.5（情绪权重相应改为 0.5），重新运行，观察最终净值变化。
2. **修改交易费用**：将交易费用从 0.1% 改为 0.3%，观察换手率高的策略净值下降了多少。
3. **添加新基准**：尝试实现"等权月度再平衡"基准（每月初重新等权分配），与买入并持有对比。

### 小数股说明（Fractional Shares）

零基础解释：小数股表示可以买入不足 1 整股的数量，例如 0.5 股。教学回测允许小数股，是为了简化资金分配逻辑。

重要说明：
- 教学回测不是 A 股真实成交规则模拟；
- A 股普通股票通常有 100 股（1 手）的交易单位限制；
- 美股等市场允许小数股交易，但取决于券商；
- Phase 1 重点是学习策略逻辑，而不是复刻完整券商撮合规则。

下一课与本课有什么关系：下一课是 Phase 1 的总结课，将通过消融实验系统比较动量策略、情绪策略和联合策略，并引入检索增强生成和 TF-IDF 概念。